In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import nltk
from nltk.tokenize import sent_tokenize

import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification)

from tqdm.auto import tqdm
import json

In [2]:
review_file = "/user/HS402/kk01697/Documents/dissertation/story-evaluation-dissertation/data/raw/goodreads/goodreads_reviews_fantasy_paranormal.json"

reviews = []
with open(review_file, "r") as f:
    for line in tqdm(f):
        reviews.append(json.loads(line))
reviews_df = pd.DataFrame(reviews)
print(reviews_df.shape)
print(reviews_df.head())

custom_dim=['Narrative Structure & Quality','Character & Emotion','Originality','Immersion','Thematic Depth','Writing Style']


0it [00:00, ?it/s]

(3424641, 11)
                            user_id   book_id  \
0  8842281e1d1347389f2ab93d60773d4d  18245960   
1  8842281e1d1347389f2ab93d60773d4d   5577844   
2  8842281e1d1347389f2ab93d60773d4d  17315048   
3  8842281e1d1347389f2ab93d60773d4d  13453029   
4  8842281e1d1347389f2ab93d60773d4d  13239822   

                          review_id  rating  \
0  dfdbb7b0eb5a7e4c26d59a937e2e5feb       5   
1  52c8ac49496c153e4a97161e36b2db55       5   
2  885c772fb033b041f42d57cef5be0a43       5   
3  46a6e1a14e8afc82d221fec0a2bd3dd0       4   
4  a582bfa8efd69d453a5a21a678046b36       3   

                                         review_text  \
0  This is a special book. It started slow for ab...   
1  A beautiful story. Neil Gaiman is truly a uniq...   
2  Mark Watney is a steely-eyed missile man. A ma...   
3  A fun fast paced book that sucks you in right ...   
4  This book has a great premise, and is full of ...   

                       date_added                    date_updated  \
0 

In [3]:
reviews_df = reviews_df[reviews_df["review_text"].str.strip().str.len() > 50].reset_index(drop=True)

records = []
for _, row in reviews_df.iterrows():
    sentences = nltk.sent_tokenize(row["review_text"])
    for idx, sent in enumerate(sentences):
        sent = sent.strip()
        if len(sent.split()) < 5:   # skip very short sentences
            continue
        records.append({
            "review_id": row["review_id"],
            "sentence_idx": idx,
            "sentence": sent,
        })

In [4]:
sentences_df = pd.DataFrame(records)
print(f"Total sentences available: {len(sentences_df)}")

sample_df = sentences_df.sample(n=3000, random_state=42).reset_index(drop=True)

for dim in custom_dim:
    sample_df[dim] = 0

sample_df.to_csv("/user/HS402/kk01697/Documents/dissertation/story-evaluation-dissertation/data/processed/annotation_sample_new.csv", index=False)
print(f"Saved {len(sample_df)} sentences to annotation_sample_new.csv")
print(sample_df.head())

Total sentences available: 26462407
Saved 3000 sentences to annotation_sample_new.csv
                          review_id  sentence_idx  \
0  451c5532a1fe2dc5e20f0dc8c11cb021             9   
1  42829c79c1a58583edceb75407270322             4   
2  6808219ea0ce09e9a1fcb40d61a0fda7             1   
3  1b809923c5cc9a7b9dccf008102a22a0             6   
4  de121319fcc2589d015651dde9e60872             2   

                                            sentence  \
0  It was a brilliant Christmas read that perfect...   
1  Our hero is a Vampire with a vengence against ...   
2  Subtlety and suspense are the key elements her...   
3  However Esme, having put her magical skills on...   
4  If you grew up reading Stephen King or you're ...   

   Narrative Structure & Quality  Character & Emotion  Originality  Immersion  \
0                              0                    0            0          0   
1                              0                    0            0          0   
2              